# Using `camstim-behavior-processing`

This library turns a raw camstim **change-detection** session
(a `*_stim.pkl` behavior file + a `*_sync.h5` sync file) into a complete,
HED-annotated **NWB** file.

It is organised in two composable layers:

1. **`load_data/`** — pure pandas/numpy. Builds the intermediate DataFrames
   (`events_df`, `trials_df`, `intervals_df`) and the running-wheel df. No NWB.
2. **`nwb/`** — takes those intermediates and assembles the NWB containers
   (trials, a flat intervals table, stimulus/movie presentations, an
   ndx-events `EventsTable`, running speed, subject + task-parameter metadata).

Two entry points tie it together:

- **`package_nwb(pkl, sync, ...)`** — the one-call path: raw files → NWB on disk.
- **`assemble_nwbfile(pkl, events_df, intervals_df, wheel_df, task_parameters)`** —
  assemble from already-built DataFrames (what this notebook demonstrates, so it
  runs without real recordings or the legacy extractor).

## Install

```bash
pip install camstim-behavior-processing
# or, from a clone of the repo:
pip install -e .
# or with uv:
uv sync
```

## 1. The one-call path (with real session files)

If you have a real `*_stim.pkl` + `*_sync.h5` on disk, this is all you need.
It builds every intermediate, assembles the full NWB, writes it as **NWB-Zarr**
(pass `fmt="hdf5"` for HDF5), and drops a BIDS-style `*.events.json` sidecar
next to it.

> **Note.** `package_nwb` calls the legacy Stage-1 extractor
> (`build_events_and_intervals`), which must be importable on `sys.path`. The
> cell below is therefore shown but **not executed** here — the rest of the
> notebook uses synthetic data so it runs anywhere.

In [ ]:
# from camstim_behavior_processing.nwb import package_nwb
#
# nwb = package_nwb(
#     "123456_stim.pkl",
#     "123456_sync.h5",
#     output_path="behavior.nwb.zarr",   # omit to just return the NWBFile
#     fmt="zarr",                         # or "hdf5"
#     metadata={"institution": "AIND", "sex": "M"},  # optional overrides
# )

### Don't know the session type? Use `package_session`

`package_nwb` is change-detection-only; a passive **SweepStim** session (natural
movie / gratings) has no `items['behavior']` and is packaged by
`package_sweepstim_nwb` instead. **`package_session`** is the type-agnostic
entry point — it inspects the pkl (`classify_sweepstim_session`) and dispatches
to the right packager for you, forwarding all the same keyword arguments. Point
it at any session in a mixed batch and it routes itself.

> Same note as above: shown but **not executed** here — it calls the legacy
> Stage-1 extractor for change-detection sessions and needs real files.

In [ ]:
# from camstim_behavior_processing import package_session
#
# # Works for BOTH change-detection and passive SweepStim sessions:
# nwb = package_session(
#     "123456_stim.pkl",
#     "123456_sync.h5",
#     output_path="behavior.nwb.zarr",   # omit to just return the NWBFile
#     fmt="zarr",                         # or "hdf5"
#     metadata={"institution": "AIND"},  # optional overrides
# )

## 2. Build synthetic intermediates (so this notebook runs standalone)

Normally `events_df` / `intervals_df` / `wheel_df` / `task_parameters` come from
`build_trials_and_events(pkl, sync)` and `compute_running_speed(...)`. Here we
hand-build tiny versions with the same schema — one go/hit trial, one
catch/correct-reject trial, a few image flashes, an omitted flash, and two
movie frames.

In [ ]:
import datetime
import numpy as np
import pandas as pd
from pynwb.file import LabMetaData

# --- events_df: one row per point event / stimulus on-offset ---
_ECOLS = ["timestamp", "event_type", "trials_id", "stimulus_presentations_id",
          "image_name", "orientation", "frame", "reward_volume",
          "lick_classification", "bout_start", "reward_type",
          "movie_frame_index", "movie_repeat"]
_erows = []
def _e(**kw):
    _erows.append({c: kw.get(c, np.nan) for c in _ECOLS})

_e(timestamp=1.0, event_type="image_onset", trials_id=0,
   stimulus_presentations_id=0, image_name="im000", frame=60)
_e(timestamp=1.25, event_type="image_offset", trials_id=0,
   stimulus_presentations_id=0, image_name="im000", frame=75)
_e(timestamp=2.0, event_type="image_onset", trials_id=0,
   stimulus_presentations_id=1, image_name="im031", frame=120)
_e(timestamp=2.0, event_type="image_change", trials_id=0,
   stimulus_presentations_id=1, image_name="im031", frame=120)
_e(timestamp=2.25, event_type="image_offset", trials_id=0,
   stimulus_presentations_id=1, image_name="im031", frame=135)
_e(timestamp=2.3, event_type="lick", trials_id=0, frame=138,
   lick_classification="hit", bout_start=True)
_e(timestamp=2.35, event_type="lick", trials_id=0, frame=140, bout_start=False)
_e(timestamp=2.4, event_type="reward", trials_id=0, frame=142,
   reward_volume=0.007, reward_type="earned")
_e(timestamp=3.0, event_type="image_omission", trials_id=0, frame=180)
_e(timestamp=10.0, event_type="movie_onset", frame=600,
   movie_frame_index=0, movie_repeat=0)
_e(timestamp=10.03, event_type="movie_offset", frame=602,
   movie_frame_index=0, movie_repeat=0)
_e(timestamp=10.03, event_type="movie_onset", frame=602,
   movie_frame_index=1, movie_repeat=0)
_e(timestamp=10.06, event_type="movie_offset", frame=604,
   movie_frame_index=1, movie_repeat=0)
events_df = pd.DataFrame(_erows)

# --- intervals_df: epochs, trials, and per-trial windows ---
_TCOLS = ["go", "catch", "auto_rewarded", "aborted", "hit", "miss",
          "false_alarm", "correct_reject", "change_time", "change_frame",
          "initial_image_name", "change_image_name", "initial_orientation",
          "change_orientation", "reward_time", "reward_volume",
          "response_time", "response_latency"]
_irows = []
def _i(start, stop, itype, label="", trials_id=np.nan, hed_string=np.nan, **kw):
    r = {c: np.nan for c in _TCOLS}
    r.update(start_time=start, stop_time=stop, interval_type=itype,
             label=label, trials_id=trials_id, hed_string=hed_string)
    r.update(kw)
    _irows.append(r)

_i(0.0, 0.5, "epoch", label="warm_up")
_i(0.5, 9.0, "epoch", label="change_detection")
_i(10.0, 11.0, "epoch", label="natural_movie_one")
_i(1.0, 3.5, "trial", trials_id=0, go=True, catch=False, auto_rewarded=False,
   aborted=False, hit=True, miss=False, false_alarm=False, correct_reject=False,
   change_time=2.0, change_frame=120, initial_image_name="im000",
   change_image_name="im031", reward_time=2.4, reward_volume=0.007,
   response_time=2.3, response_latency=0.3)
_i(1.9, 2.1, "change_window", trials_id=0, hed_string="Def/change_window")
_i(2.0, 2.75, "response_window", trials_id=0, hed_string="Def/response_window")
_i(4.0, 6.5, "trial", trials_id=1, go=False, catch=True, auto_rewarded=False,
   aborted=False, hit=False, miss=False, false_alarm=False, correct_reject=True,
   initial_image_name="im000", change_image_name="im000", reward_volume=0.0)
intervals_df = pd.DataFrame(_irows)

# --- the loaded pkl (identity + params) ---
pkl = {"start_time": datetime.datetime(2024, 1, 2, 3, 4, 5),
       "items": {"behavior": {"params": {
           "mouse_id": "123456", "stage": "OPHYS_1_images_A",
           "warm_up_trials": 1}}}}

# --- running-wheel df (normally from compute_running_speed) ---
t = np.linspace(0, 11, 660)
wheel_df = pd.DataFrame(
    {"speed": np.zeros_like(t), "dx": np.zeros_like(t),
     "v_sig": np.ones_like(t), "v_in": np.full_like(t, 5.0)},
    index=pd.Index(t, name="timestamps"))

# --- task parameters (normally a ChangeDetectionTaskParameters lab-meta) ---
task_parameters = LabMetaData(name="task_parameters")

events_df.head()

## 3. Assemble the complete NWB

`assemble_nwbfile` wires everything together and returns an
`ndx_events.NdxEventsNWBFile`.

In [ ]:
from camstim_behavior_processing.nwb import assemble_nwbfile

nwb = assemble_nwbfile(
    pkl, events_df, intervals_df, wheel_df, task_parameters,
    metadata={"institution": "Allen Institute for Neural Dynamics"},
)
nwb

## 4. Inspect what got built

In [ ]:
print("subject:        ", nwb.subject.subject_id, "/", nwb.subject.species)
print("lab metadata:   ", list(nwb.lab_meta_data))
print("interval tables:", list(nwb.intervals))
print("events rows:    ", len(nwb.get_events_table("events")))
print("processing:     ", list(nwb.processing))
print("acquisition:    ", list(nwb.acquisition))

### Trials table

In [ ]:
trials = nwb.trials.to_dataframe()
trials[["start_time", "stop_time", "go", "catch", "hit", "correct_reject",
        "change_time", "response_latency", "warm_up", "epoch_name", "HED"]]

### Discrete events (ndx-events `EventsTable`)\n\nOnly discrete events survive here (lick / reward / image_change / image_omission); image and movie on/offsets live on the interval tables, and `miss` is a trial outcome.

In [ ]:
et = nwb.get_events_table("events").to_dataframe()
et[["timestamp", "event_type", "lick_classification", "reward_type",
    "lick_bouts", "reward_volume"]]

### Stimulus presentations (one row per flash + omitted slot)

In [ ]:
sp = nwb.intervals["stimulus_presentations"].to_dataframe()
sp[["start_time", "stop_time", "image_name", "is_change", "omitted",
    "epoch_name", "HED"]]

### Canonical flat intervals table (every interval type)

In [ ]:
iv = nwb.intervals["intervals"].to_dataframe()
iv[["start_time", "stop_time", "interval_type", "label", "trials_id",
    "stimulus_presentations_id"]]

### Running speed

In [ ]:
speed = nwb.processing["running"]["speed"]
print("speed samples:", speed.data[:].shape, "unit:", speed.unit)

## 5. Write to disk and read it back

The library writes **NWB-Zarr** by default (via `hdmf-zarr`). Here we write with
`NWBZarrIO` directly and read the file back to confirm the round-trip.

In [ ]:
import tempfile
from pathlib import Path
from hdmf_zarr.nwb import NWBZarrIO

out_dir = Path(tempfile.mkdtemp())
out_path = out_dir / "behavior.nwb.zarr"
with NWBZarrIO(str(out_path), mode="w") as io:
    io.write(nwb)

with NWBZarrIO(str(out_path), mode="r") as io:
    reloaded = io.read()
    print("read back:", len(reloaded.trials), "trials,",
          len(reloaded.get_events_table("events")), "events")

The BIDS-style events sidecar (column descriptions + HED) is produced by
`build_events_sidecar()`; `package_nwb` writes it next to the NWB automatically.
Here is what it contains for one column:

In [ ]:
from camstim_behavior_processing.nwb import build_events_sidecar

sidecar = build_events_sidecar()
sidecar["event_type"]

## 6. Just the Layer-1 DataFrames (no NWB)

If you only want the analysis tables from a real session, call the loaders
directly — no NWB dependency is triggered. (Shown, not executed: needs real
files + the legacy extractor.)

```python
from camstim_behavior_processing import (
    build_trials_and_events, compute_running_speed,
)
from camstim_behavior_processing.load_data import load_stim_pkl

built = build_trials_and_events("123456_stim.pkl", "123456_sync.h5")
events_df = built["events_df"]
intervals_df = built["intervals_df"]

pkl = load_stim_pkl("123456_stim.pkl")
wheel_df = compute_running_speed(
    pkl, built["timestamp_data"]["stim_vsync_fall"],
)
```